# Dubai Rental Market Analysis (2020–2026)
### What's driving rental growth across Dubai's key districts?

**Author:** Bianca Khambatta  
**Dataset:** Dubai Real Estate: Sales, Off-Plan & Rentals (Kaggle — sergionefedov)  
**Last updated:** June 2026

---

**A note on the data:** Listing-level records are generated via a hedonic pricing model anchored to real DLD/Property Finder base prices per zone, real CBUAE mortgage rate timelines, and real Dubai Metro coordinates. This is standard methodology in real estate econometrics — the same approach used by central banks and property research firms to produce market indices where transaction-level data is incomplete or proprietary.

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# paths
DATA = r"C:\Users\biakh\OneDrive\Desktop\dubai-rental-market-analysis\dataraw"

# consistent plot style throughout
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'sans-serif'
PALETTE = ['#2C5F8A', '#E8873A', '#3A9E6F', '#C0392B', '#8E44AD', '#17A589']

print("Libraries loaded.")

---
## 1. Load & First Look

Load all five files and do a quick sanity check on each before touching anything.

In [ ]:
import os

rentals      = pd.read_csv(os.path.join(DATA, 'rentals.csv'))
sales        = pd.read_csv(os.path.join(DATA, 'secondary_sales.csv'))
monthly      = pd.read_csv(os.path.join(DATA, 'area_prices_monthly.csv'))
metro        = pd.read_csv(os.path.join(DATA, 'metro_stations.csv'))
offplan      = pd.read_csv(os.path.join(DATA, 'off_plan.csv'))

for name, df in [('rentals', rentals), ('sales', sales), 
                  ('monthly', monthly), ('metro', metro), ('offplan', offplan)]:
    print(f"{name:10s} — {df.shape[0]:,} rows | {df.shape[1]} cols")

In [ ]:
# quick look at the rentals file since that's our primary
rentals.head(3)

In [ ]:
rentals.dtypes

In [ ]:
# missing values — check before doing anything else
missing = rentals.isnull().sum()
missing[missing > 0]

---
## 2. Cleaning & Preprocessing

Parse dates, drop duplicates, handle nulls, and flag anomalies. Document every decision.

In [ ]:
# --- rentals ---
rentals['date'] = pd.to_datetime(rentals['date_listed'], errors='coerce')
rentals['year'] = rentals['date'].dt.year
rentals['month'] = rentals['date'].dt.month
rentals['quarter'] = rentals['date'].dt.to_period('Q').astype(str)
rentals['year_month'] = rentals['date'].dt.to_period('M').astype(str)

# drop rows where rent or date is missing — can't do time-series without these
before = len(rentals)
rentals.dropna(subset=['annual_rent_usd', 'date'], inplace=True)
print(f"Dropped {before - len(rentals)} rows with missing rent or date.")

# remove statistical outliers in rent — anything beyond 3 std devs is almost certainly a data artefact
z = np.abs(stats.zscore(rentals['annual_rent_usd']))
before = len(rentals)
rentals = rentals[z < 3]
print(f"Removed {before - len(rentals)} rent outliers (>3 SD).")

# --- monthly time series ---
monthly['date'] = pd.to_datetime(monthly['year_month'] if 'year_month' in monthly.columns 
                                  else monthly.iloc[:, 0], errors='coerce')
monthly.sort_values('date', inplace=True)

# --- sales ---
sales['date'] = pd.to_datetime(sales['date_listed'], errors='coerce')
sales['year'] = sales['date'].dt.year
sales['quarter'] = sales['date'].dt.to_period('Q').astype(str)

print("\nCleaning complete.")
print(f"Rentals: {len(rentals):,} records | {rentals['year'].min()}–{rentals['year'].max()}")

---
## 3. Feature Engineering

Build the variables we actually need for the economic analysis — things that aren't in the raw data but emerge from it.

In [ ]:
# --- market segment: luxury vs budget ---
# Thresholds anchored to Dubai market reality:
#   Budget  = studios & 1BR under $20K/yr (AED ~73K) 
#   Mid     = everything in between
#   Luxury  = 3BR+ or villas above $60K/yr (AED ~220K)

def segment(row):
    if row['annual_rent_usd'] < 20000 and row.get('bedrooms', 1) <= 1:
        return 'Budget'
    elif row['annual_rent_usd'] > 60000 or row.get('property_category', '') == 'villa':
        return 'Luxury'
    else:
        return 'Mid-market'

rentals['segment'] = rentals.apply(segment, axis=1)
print(rentals['segment'].value_counts())

In [ ]:
# --- YoY growth per community ---
# calculate median rent per community per year, then compute % change

community_yr = (rentals
    .groupby(['community', 'year'])['annual_rent_usd']
    .median()
    .reset_index()
    .rename(columns={'annual_rent_usd': 'median_rent'}))

community_yr['yoy_growth'] = (community_yr
    .groupby('community')['median_rent']
    .pct_change() * 100)

community_yr.head(10)

In [ ]:
# --- indexed price (2020 Q1 = 100) ---
# normalise rent to a base period so we can compare growth across segments

quarterly = (rentals
    .groupby(['quarter', 'segment'])['annual_rent_usd']
    .median()
    .reset_index()
    .rename(columns={'annual_rent_usd': 'median_rent'}))

base = quarterly[quarterly['quarter'] == '2020Q1'][['segment', 'median_rent']].rename(
    columns={'median_rent': 'base_rent'})

quarterly = quarterly.merge(base, on='segment', how='left')
quarterly['price_index'] = (quarterly['median_rent'] / quarterly['base_rent']) * 100

print("Indexed price series built.")
quarterly.head()

In [ ]:
# --- metro proximity flag ---
# walkable = under 10 minutes on foot; premium = under 5 min

if 'metro_distance_min' in rentals.columns and 'metro_distance_type' in rentals.columns:
    rentals['metro_walkable'] = (
        (rentals['metro_distance_min'] <= 10) & 
        (rentals['metro_distance_type'] == 'walk')
    )
    rentals['metro_premium'] = (
        (rentals['metro_distance_min'] <= 5) & 
        (rentals['metro_distance_type'] == 'walk')
    )
    print(f"Walkable to metro: {rentals['metro_walkable'].sum():,} listings")
    print(f"Premium proximity (<5 min): {rentals['metro_premium'].sum():,} listings")

---
## 4. Macroeconomic Timeline

Before any community-level analysis, establish the market-wide trend and annotate the key events we know happened. This gives us the economic narrative the rest of the analysis hangs off.

In [ ]:
# quarterly median rent — market wide
market_trend = (rentals
    .groupby('quarter')['annual_rent_usd']
    .median()
    .reset_index()
    .rename(columns={'annual_rent_usd': 'median_rent'}))

market_trend['quarter_dt'] = pd.PeriodIndex(market_trend['quarter'], freq='Q').to_timestamp()
market_trend.sort_values('quarter_dt', inplace=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(market_trend['quarter_dt'], market_trend['median_rent'], 
        color=PALETTE[0], linewidth=2.5, zorder=3)
ax.fill_between(market_trend['quarter_dt'], market_trend['median_rent'], 
                alpha=0.08, color=PALETTE[0])

# annotate macro events
events = {
    '2020Q2': ('COVID\nlockdown', -4000),
    '2021Q4': ('Expo 2020\nopens', 2000),
    '2022Q1': ('Russian/CIS\ncapital influx', 2000),
    '2022Q4': ('Golden Visa\nexpansion', -4000),
    '2023Q3': ('Market\npeak', 2000),
    '2024Q2': ('Rate\ncooling', -4000),
}

for qtr, (label, offset) in events.items():
    try:
        x = pd.Period(qtr, freq='Q').to_timestamp()
        y = market_trend.loc[market_trend['quarter'] == qtr, 'median_rent'].values
        if len(y):
            ax.axvline(x, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
            ax.annotate(label, xy=(x, y[0]), xytext=(x, y[0] + offset),
                        fontsize=8, ha='center', color='#444',
                        arrowprops=dict(arrowstyle='->', color='grey', lw=0.8))
    except:
        pass

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Dubai Rental Market — Quarterly Median Rent (2020–2026)', 
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Median Annual Rent (USD)')
plt.tight_layout()
plt.savefig('market_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. K-Shaped Recovery Analysis

The central hypothesis: did COVID produce a bifurcated recovery where the luxury segment bounced back faster and harder than the budget segment — a so-called K-shaped trajectory? We test this using indexed prices (2020 Q1 = 100) across our three segments.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

segment_order = ['Luxury', 'Mid-market', 'Budget']
colors = {'Luxury': PALETTE[0], 'Mid-market': PALETTE[2], 'Budget': PALETTE[1]}

for seg in segment_order:
    sub = quarterly[quarterly['segment'] == seg].copy()
    sub['q_dt'] = pd.PeriodIndex(sub['quarter'], freq='Q').to_timestamp()
    sub.sort_values('q_dt', inplace=True)
    ax.plot(sub['q_dt'], sub['price_index'], 
            label=seg, color=colors[seg], linewidth=2.2)

ax.axhline(100, color='grey', linestyle=':', linewidth=1, label='2020 Q1 baseline')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-09-01'), 
           alpha=0.07, color='red', label='COVID impact window')

ax.set_title('K-Shaped Recovery: Price Index by Market Segment (2020 Q1 = 100)', 
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Price Index')
ax.set_xlabel('')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('k_shaped_recovery.png', dpi=150, bbox_inches='tight')
plt.show()

# print the actual index values at peak and current for each segment
print("\nSegment performance summary:")
for seg in segment_order:
    sub = quarterly[quarterly['segment'] == seg]
    peak = sub['price_index'].max()
    current = sub.iloc[-1]['price_index']
    print(f"  {seg:12s} — Peak index: {peak:.1f} | Current: {current:.1f}")

---
## 6. Mortgage Rate vs Rental Growth

When buying becomes expensive (high mortgage rates), demand shifts into the rental market — pushing rents up. We test whether the CBUAE mortgage rate (which tracks the US Fed) correlates with rental price acceleration.

In [ ]:
# we need the mortgage rate column from the monthly time series
print(monthly.columns.tolist())
monthly.head(3)

In [ ]:
# get market-wide median rent by month
monthly_rent = (rentals
    .groupby('year_month')['annual_rent_usd']
    .median()
    .reset_index()
    .rename(columns={'annual_rent_usd': 'median_rent'}))

monthly_rent['date'] = pd.to_datetime(monthly_rent['year_month'])
monthly_rent.sort_values('date', inplace=True)
monthly_rent['rent_yoy'] = monthly_rent['median_rent'].pct_change(12) * 100

# try to find mortgage rate column in monthly file
rate_col = [c for c in monthly.columns if 'mortgage' in c.lower() or 'rate' in c.lower()]
print("Rate columns found:", rate_col)

In [ ]:
if rate_col:
    rc = rate_col[0]
    
    # merge monthly rent with mortgage rate
    date_col = [c for c in monthly.columns if 'date' in c.lower() or 'month' in c.lower()][0]
    rate_df = monthly[['date', rc]].copy() if 'date' in monthly.columns else monthly[[date_col, rc]].copy()
    rate_df.columns = ['date', 'mortgage_rate']
    rate_df['date'] = pd.to_datetime(rate_df['date'])
    
    merged = monthly_rent.merge(rate_df, on='date', how='inner').dropna()

    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax2 = ax1.twinx()

    ax1.plot(merged['date'], merged['median_rent'], color=PALETTE[0], 
             linewidth=2, label='Median annual rent (USD)')
    ax2.plot(merged['date'], merged['mortgage_rate'], color=PALETTE[1], 
             linewidth=1.8, linestyle='--', label='Mortgage rate (%)')

    ax1.set_ylabel('Median Annual Rent (USD)', color=PALETTE[0])
    ax2.set_ylabel('UAE Mortgage Rate (%)', color=PALETTE[1])
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc='upper left')

    ax1.set_title('Rental Prices vs UAE Mortgage Rate (2020–2026)', 
                  fontsize=13, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig('mortgage_vs_rent.png', dpi=150, bbox_inches='tight')
    plt.show()

    # pearson correlation
    r, p = pearsonr(merged['mortgage_rate'], merged['median_rent'])
    print(f"\nPearson r (mortgage rate vs rent): {r:.3f} | p-value: {p:.4f}")
    print("Interpretation:", "Significant positive correlation" if p < 0.05 and r > 0 
          else "Significant negative correlation" if p < 0.05 
          else "No significant linear relationship")

---
## 7. Leading vs Lagging Districts

Do some communities show price increases before the broader market moves? If so, they're leading indicators — useful for investors and policy analysts. We identify them by comparing each community's growth trajectory against the market-wide quarterly trend.

In [ ]:
# communities with enough data to be meaningful (at least 3 years of records)
min_years = 3
community_year_counts = rentals.groupby('community')['year'].nunique()
valid_communities = community_year_counts[community_year_counts >= min_years].index
print(f"{len(valid_communities)} communities with {min_years}+ years of data")

In [ ]:
# market-wide quarterly median (our benchmark)
market_q = (rentals
    .groupby('quarter')['annual_rent_usd']
    .median()
    .rename('market_median')
    .reset_index())

# community-level quarterly median
comm_q = (rentals[rentals['community'].isin(valid_communities)]
    .groupby(['community', 'quarter'])['annual_rent_usd']
    .median()
    .reset_index()
    .rename(columns={'annual_rent_usd': 'comm_median'}))

comm_q = comm_q.merge(market_q, on='quarter')
comm_q['premium_over_market'] = ((comm_q['comm_median'] - comm_q['market_median']) 
                                   / comm_q['market_median'] * 100)

# rank communities by cumulative YoY growth 2020-2023
growth_rank = (community_yr[community_yr['community'].isin(valid_communities)]
    .groupby('community')['yoy_growth']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'yoy_growth': 'avg_yoy_growth'}))

top10 = growth_rank.head(10)
bottom10 = growth_rank.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].barh(top10['community'], top10['avg_yoy_growth'], color=PALETTE[2])
axes[0].set_title('Top 10 Leading Communities\n(Avg YoY Rental Growth)', fontweight='bold')
axes[0].set_xlabel('Average YoY Growth (%)')
axes[0].invert_yaxis()

axes[1].barh(bottom10['community'], bottom10['avg_yoy_growth'], color=PALETTE[3])
axes[1].set_title('Bottom 10 Lagging Communities\n(Avg YoY Rental Growth)', fontweight='bold')
axes[1].set_xlabel('Average YoY Growth (%)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('leading_lagging_communities.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Price Elasticity by Bedroom Category

Do larger units absorb price shocks differently than studios and 1-beds? We compare how each bedroom category responded to the key macro shocks: COVID dip (2020), rally peak (2023), and rate cooling (2024).

In [ ]:
if 'bedrooms' in rentals.columns:
    
    # cap bedrooms at 4+ for cleaner grouping
    rentals['bedroom_grp'] = rentals['bedrooms'].clip(upper=4)
    bedroom_labels = {0: 'Studio', 1: '1 BR', 2: '2 BR', 3: '3 BR', 4: '4+ BR'}
    rentals['bedroom_label'] = rentals['bedroom_grp'].map(bedroom_labels)

    bed_quarter = (rentals
        .groupby(['quarter', 'bedroom_label'])['annual_rent_usd']
        .median()
        .reset_index()
        .rename(columns={'annual_rent_usd': 'median_rent'}))

    # index to 2020 Q1
    base_bed = (bed_quarter[bed_quarter['quarter'] == '2020Q1']
        [['bedroom_label', 'median_rent']]
        .rename(columns={'median_rent': 'base'}))
    
    bed_quarter = bed_quarter.merge(base_bed, on='bedroom_label')
    bed_quarter['index'] = bed_quarter['median_rent'] / bed_quarter['base'] * 100
    bed_quarter['q_dt'] = pd.PeriodIndex(bed_quarter['quarter'], freq='Q').to_timestamp()

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, label in enumerate(bedroom_labels.values()):
        sub = bed_quarter[bed_quarter['bedroom_label'] == label].sort_values('q_dt')
        if not sub.empty:
            ax.plot(sub['q_dt'], sub['index'], label=label, 
                    color=PALETTE[i % len(PALETTE)], linewidth=1.8)

    ax.axhline(100, color='grey', linestyle=':', linewidth=1)
    ax.set_title('Price Index by Bedroom Category (2020 Q1 = 100)', 
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('Price Index')
    ax.legend(frameon=False, ncol=5)
    plt.tight_layout()
    plt.savefig('bedroom_elasticity.png', dpi=150, bbox_inches='tight')
    plt.show()

    # COVID trough and 2023 peak by bedroom type
    print("\nShock response by bedroom type:")
    for label in bedroom_labels.values():
        sub = bed_quarter[bed_quarter['bedroom_label'] == label]
        if not sub.empty:
            trough = sub['index'].min()
            peak = sub['index'].max()
            print(f"  {label:8s} — COVID trough: {trough:.1f} | Rally peak: {peak:.1f} | Range: {peak-trough:.1f} pts")

---
## 9. Metro Infrastructure Premium

Does proximity to a Dubai Metro station command a measurable rental premium? We compare median rents for walkable vs non-walkable listings, and test whether this premium has grown as the Route 2020 extension embedded into the market post-2021.

In [ ]:
if 'metro_walkable' in rentals.columns:
    
    metro_comp = (rentals
        .groupby(['year', 'metro_walkable'])['annual_rent_usd']
        .median()
        .unstack()
        .rename(columns={False: 'Not walkable', True: 'Walkable (<10 min)'})
        .reset_index())

    metro_comp['premium_pct'] = ((metro_comp['Walkable (<10 min)'] - metro_comp['Not walkable']) 
                                  / metro_comp['Not walkable'] * 100)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    metro_comp.plot(x='year', y=['Not walkable', 'Walkable (<10 min)'], 
                    ax=ax1, color=[PALETTE[1], PALETTE[0]], linewidth=2)
    ax1.set_title('Median Rent: Walkable vs Non-Walkable\nto Metro Station', fontweight='bold')
    ax1.set_ylabel('Median Annual Rent (USD)')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax1.legend(frameon=False)
    ax1.set_xlabel('')

    ax2.bar(metro_comp['year'], metro_comp['premium_pct'], color=PALETTE[2], alpha=0.85)
    ax2.axhline(0, color='grey', linewidth=0.8)
    ax2.set_title('Metro Walkability Premium\nby Year (%)', fontweight='bold')
    ax2.set_ylabel('Rental Premium (%)')
    ax2.set_xlabel('')

    plt.tight_layout()
    plt.savefig('metro_premium.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nMetro premium by year:")
    print(metro_comp[['year', 'premium_pct']].to_string(index=False))

---
## 10. Freehold vs Non-Freehold Pricing

Freehold areas allow foreign nationals to own property outright — a critical distinction in Dubai's market. We test whether freehold designation commands a rental premium, and whether that premium has shifted as foreign capital inflows accelerated post-2022.

In [ ]:
if 'is_freehold' in rentals.columns:
    
    fh = (rentals
        .groupby(['year', 'is_freehold'])['annual_rent_usd']
        .median()
        .unstack()
        .rename(columns={False: 'Non-freehold', True: 'Freehold'})
        .reset_index())

    fh['premium_pct'] = (fh['Freehold'] - fh['Non-freehold']) / fh['Non-freehold'] * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(fh['year'] - 0.2, fh['Freehold'], 0.35, label='Freehold', color=PALETTE[0], alpha=0.85)
    ax.bar(fh['year'] + 0.2, fh['Non-freehold'], 0.35, label='Non-freehold', color=PALETTE[1], alpha=0.85)
    ax.set_title('Median Rent: Freehold vs Non-Freehold Areas (2020–2026)', fontweight='bold')
    ax.set_ylabel('Median Annual Rent (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(frameon=False)
    ax.set_xlabel('')
    plt.tight_layout()
    plt.savefig('freehold_premium.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nFreehold premium over time:")
    print(fh[['year', 'premium_pct']].to_string(index=False))

---
## 11. Demand-Supply Proxy: Rental Contract Frequency

In the absence of direct supply data, contract volume serves as a demand proxy. Communities where transaction volume grew faster than rent may signal undersupply — where demand is outpacing available stock. This section identifies demand-supply imbalances at the community level.

In [ ]:
# volume by community and year
volume = (rentals[rentals['community'].isin(valid_communities)]
    .groupby(['community', 'year'])
    .size()
    .reset_index(name='contract_volume'))

# combine with rent growth
vol_rent = volume.merge(community_yr, on=['community', 'year'])

# for each community: correlation between volume growth and rent growth
vol_rent['vol_yoy'] = vol_rent.groupby('community')['contract_volume'].pct_change() * 100
vol_rent_clean = vol_rent.dropna(subset=['vol_yoy', 'yoy_growth'])

# scatter: volume growth vs rent growth
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(vol_rent_clean['vol_yoy'], vol_rent_clean['yoy_growth'], 
           alpha=0.4, color=PALETTE[0], s=25)

# regression line
m, b, r, p, _ = stats.linregress(vol_rent_clean['vol_yoy'], vol_rent_clean['yoy_growth'])
x_line = np.linspace(vol_rent_clean['vol_yoy'].min(), vol_rent_clean['vol_yoy'].max(), 100)
ax.plot(x_line, m * x_line + b, color=PALETTE[3], linewidth=1.5, 
        label=f'OLS fit (r={r:.2f}, p={p:.3f})')

ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
ax.axvline(0, color='grey', linewidth=0.8, linestyle=':')
ax.set_xlabel('Contract Volume YoY Growth (%)')
ax.set_ylabel('Rental Price YoY Growth (%)')
ax.set_title('Demand Proxy vs Price Growth\n(Community-Year observations)', fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('demand_supply_proxy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOLS: volume growth explains {r**2*100:.1f}% of variance in rent growth")

---
## 12. Correlation Heatmap — What Actually Drives Rent?

Pull together all numeric variables and run a full correlation matrix to see which property characteristics most strongly predict rental price. This surfaces the structural drivers rather than just the macro ones.

In [ ]:
numeric_cols = ['annual_rent_usd', 'area_sqft', 'bedrooms', 'floor', 
                'total_floors', 'parking_spaces', 'metro_distance_min',
                'to_burj_khalifa_km', 'mortgage_rate_at_listing']

available = [c for c in numeric_cols if c in rentals.columns]
corr_data = rentals[available].dropna()

corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle only
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Matrix — Rental Price Drivers', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# top correlates with rent
print("\nTop correlates with annual_rent_usd:")
print(corr_matrix['annual_rent_usd'].drop('annual_rent_usd').sort_values(key=abs, ascending=False))

---
## 13. Summary of Findings

Fill this section in once you've run all the cells above and seen the actual numbers.

### Key findings

*(Update these with your actual output values after running the notebook)*

**1. Macro trend**  
Dubai's rental market grew from a median of $X/yr in Q1 2020 to $X/yr by Q4 2023 — a X% increase over the period. The COVID dip in Q2 2020 was brief (~X% below baseline) and fully recovered by Q[X] 2021.

**2. K-shaped recovery confirmed / not confirmed**  
The luxury segment peaked at index X (vs 2020 Q1 baseline of 100), while the budget segment reached only X — a X-point divergence. This [supports / does not support] the K-shaped recovery hypothesis.

**3. Mortgage rate relationship**  
Pearson r between mortgage rate and median rent = X (p = X). [Interpretation based on your output.]

**4. Leading communities**  
The top 3 leading communities by average YoY rental growth were [X], [X], and [X], each outperforming the market by X–X percentage points. Notably, [community] showed accelerating growth from [date], approximately [X] quarters ahead of the broader market.

**5. Bedroom elasticity**  
4+ BR units showed the widest shock range (X pts from trough to peak), while studios were most resilient during the COVID dip (trough index: X vs market average X). Larger units absorbed macro shocks more dramatically in both directions.

**6. Metro premium**  
Walkable proximity to a Metro station commanded a X% rental premium in 2020, rising to X% by 2024 — suggesting the Route 2020 extension embedded into pricing over time.

**7. Strongest structural drivers of rent**  
Based on the correlation matrix, the variables most strongly associated with higher rent were: [X] (r=X), [X] (r=X), [X] (r=X). Distance to Burj Khalifa as a city-centre proxy showed [direction] correlation (r=X).

---
*Analysis by Bianca Khambatta | Dubai, UAE | 2026*  
*Dataset: Dubai Real Estate Sales, Off-Plan & Rentals (Kaggle — Apache 2.0 License)*